# Search Service playground

Domain Agent가 사용하는 결정론적 `SearchService`를 직접 호출해 검색 결과를 확인한다.

준비 사항:

- 저장소 루트의 `.env`에 `CLOVASTUDIO_*`, `QDRANT_*` 설정하기
- 대상 Qdrant collection에 문서 인덱스가 적재되어 있는지 확인하기
- 저장소 루트에서 `uv run --locked --group notebook jupyter lab` 실행하기

노트북 출력에는 검색 문서 내용이 남을 수 있으므로 커밋 전에 모든 출력을 제거한다.

In [ ]:
from pension_agent.agent.contracts import Permission
from pension_agent.agent.search import SearchRequest, SearchService
from pension_agent.config import (
    BGE_M3_EMBEDDING_CONFIG,
    ClovaStudioConnection,
    QdrantConnection,
)
from pension_agent.retrieval import (
    create_async_qdrant_client,
    create_async_qdrant_retriever,
    create_clova_query_embedder,
)

## 연결과 Search Service 조립

이 셀을 다시 실행하면 기존 Qdrant client를 먼저 닫는다.

In [ ]:
existing_qdrant_client = globals().get("qdrant_client")
if existing_qdrant_client is not None:
    await existing_qdrant_client.close()

clova_connection = ClovaStudioConnection()
qdrant_connection = QdrantConnection()
embedder = create_clova_query_embedder(
    config=BGE_M3_EMBEDDING_CONFIG,
    connection=clova_connection,
)
qdrant_client = create_async_qdrant_client(qdrant_connection)
retriever = create_async_qdrant_retriever(
    qdrant_client,
    connection=qdrant_connection,
)
search_service = SearchService(embedder=embedder, retriever=retriever)

{
    "qdrant_url": qdrant_connection.url,
    "collection": qdrant_connection.collection,
    "embedding_model": BGE_M3_EMBEDDING_CONFIG.model,
}

## 전체 문서 검색

`POLICY`와 `TAX_PAYOUT`은 연금 참고 문서, `PRODUCT`는 투자설명서만 검색한다.

In [ ]:
request = SearchRequest(
    objective="IRP 계좌 이전 절차와 필요한 조건",
    expand_neighbors=True,
)
result = await search_service.search(
    request,
    permission=Permission.POLICY,
)
result.model_dump()

## 문서 범위를 제한한 검색

Qdrant payload의 `source_file_name`과 정확히 같은 파일명을 사용한다.

In [ ]:
request = SearchRequest(
    objective="연금 수령 시 세율",
    source_file_name="원본-파일명.pdf",
)
result = await search_service.search(
    request,
    permission=Permission.TAX_PAYOUT,
)
result.model_dump()

## 정리

실험을 마치면 Qdrant client를 닫는다.

In [ ]:
await qdrant_client.close()